# Single-channel beam fit at 156.25 MHz

**Question.** How well can one channel be fitted with spherical harmonics alone ($\ell_{\max}=8$), with no HFSS prior and no bridging across frequency? The score table compares it with HFSS and with the reference beam, `derived/beam/empirical_raster_v0013` (DPSS and PCA spectral bases, both arms, all 64 teeth).

**Model.** $d_i \simeq g\,\left|\sum_{\ell m}\mathbf c_{\ell m}\,Y_{\ell m}(\Omega_i)\cdot\mathbf e_i\right|^2$, the model of `eigsep_data.beam_mapping.JointBeamFit` with a one-tooth `ToothData` and a constant spectral basis. The $\mathbf c_{\ell m}$ are complex Cartesian 3-vectors on real spherical harmonics. $\mathbf e_i$ is the transmitter arm's field projected transverse to the line of sight. The prior weight is zero. With one arm only the field component along $\mathbf e_i$ is measured, so the orthogonal component, and with it the split between $g$ and the field's normalization, is unconstrained; the overall phase is unobservable.

**Inputs, all from v0013** (checked against its `provenance.json`):
- The 07-17 20:26:00–21:28:40 UTC raster (box-air, RFANT), pointing from `curation/pointing_table.parquet` (commanded azimuth, IMU elevation), the local-DPSS background under each tooth, and the masks (gap-ratio dropouts, reversal glitches, `raster_flags_v0001` sample and tooth flags): the `data`, `good` and `split` arrays of `dpss/diagnostics.npz`.
- **Geometry:** antenna and transmitter from geometry release `imgs/fits/v0004_marjum_geometry`; highline mount convention $\psi$, az offset, polarization centre and the five fitted corrections (az/el zero, polarization, transmitter offsets) from v0013.
- **Split:** v0013's 12° stripes of commanded azimuth (train / validation / test).

The fit starts from the normalized HFSS field at this tooth projected onto the harmonics. A training-stripe fit gives the held-out scores; an all-sample fit gives the maps.

In [ ]:
from pathlib import Path
import hashlib
import json
import os

os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('JAX_ENABLE_X64', 'true')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import display

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)

import eigsep_data
import eigsep_data.beam_mapping as bm
from eigsep_data.beam_sim import read_beam

if os.environ.get('EIGSEP_CAMPAIGN_ROOT'):
    CAMPAIGN = Path(os.environ['EIGSEP_CAMPAIGN_ROOT'])
else:
    CAMPAIGN = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'marjum-2026-07/data').is_dir()) / 'marjum-2026-07'
eigsep_data.set_campaign_root(CAMPAIGN)
WORKSPACE = CAMPAIGN.parent

PRODUCT = CAMPAIGN / 'derived/beam/empirical_raster_v0013'   # the reference beam
CHANNEL = 640                                                # 156.25 MHz, an HFSS slice
LMAX = 8


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


prov = json.loads((PRODUCT / 'provenance.json').read_text())
for name in ('dpss/diagnostics.npz', 'pca/diagnostics.npz'):
    assert sha256(PRODUCT / name) == prov['artifact_sha256'][name], name
release_path = WORKSPACE / prov['geometry_release']['path']
assert sha256(release_path) == prov['input_sha256'][prov['geometry_release']['path']], 'release changed'
release = json.loads(release_path.read_text())
ANT = np.asarray(release['antenna_91m_era']['position_enu_m'], float)
TX = np.asarray(release['transmitter']['recommended_for_propagation']['position_enu_m'], float)
print('beam product %s: %s; geometry release %s; pointing %s'
      % (PRODUCT.name, prov['status'], release['release'], prov['pointing']))
print('antenna %s, transmitter %s (m, UTM raster grid)' % (np.round(ANT, 2), np.round(TX, 2)))

In [ ]:
# --- the tooth, its masks and split, and the v0013 models at this tooth --------
diag = {}
for kind in ('dpss', 'pca'):
    with np.load(PRODUCT / kind / 'diagnostics.npz') as z:
        diag[kind] = {key: z[key] for key in z.files}
d = diag['dpss']
assert np.array_equal(d['data'], diag['pca']['data']) and np.array_equal(d['good'], diag['pca']['good'])
i = int(np.flatnonzero(d['channels'] == CHANNEL)[0])
PARAMS = d['params']
MODEL = bm.TxGeometryModel(float(d['psi_deg']), ANT, TX, alpha0_deg=float(d['alpha0_deg']),
                           az_offset_deg=float(d['az_offset_deg']))
DATA1 = bm.ToothData(t=d['t'], az_deg=d['az_table'], el_deg=d['el'], data=d['data'][i:i + 1],
                     good=d['good'][i:i + 1], channels=d['channels'][i:i + 1],
                     freqs_mhz=d['freqs'][i:i + 1], split=d['split'])
assert DATA1.arms[0] == d['arms'][i]
DATA, GOOD, SPLIT = DATA1.data[0], DATA1.good[0], DATA1.split
TRAIN = GOOD & (SPLIT == 0)
print('channel %d (%.3f MHz), arm %d; %d good samples (train/val/test %s)'
      % (CHANNEL, d['freqs'][i], DATA1.arms[0], GOOD.sum(), np.bincount(SPLIT[GOOD], minlength=3)))
print('geometry: psi %.3f, az offset %+.1f, alpha0 %.1f; corrections %s'
      % (d['psi_deg'], d['az_offset_deg'], d['alpha0_deg'],
         {k: round(v, 3) for k, v in zip(bm.PARAM_NAMES, PARAMS)}))

# HFSS at this tooth, normalized as the package does; reproduce v0013's HFSS baseline.
hfields, _, hfreqs = read_beam(drop_last=False)
FIELD = bm.tooth_fields(hfields, hfreqs, DATA1.freqs_mhz)                 # (1, 3, npix)
hpow = bm.hfss_power(FIELD, DATA1, MODEL, PARAMS)
HFSS = bm.tooth_gains(hpow, DATA1.data, DATA1.good, SPLIT == 0)[:, None] * hpow
err = np.max(np.abs(HFSS[0] - d['hfss'][i])) / np.max(np.abs(d['hfss'][i]))
print('HFSS baseline reproduced from v0013 inputs: max relative difference %.1e' % err)
assert err < 1e-6
HFSS = HFSS[0]

In [ ]:
# --- fit: harmonic coefficients only, no prior, through JointBeamFit ------------
class ConstantSpectrum:
    # A one-mode spectral basis: the single tooth is its own coefficient set.
    nmodes = 1

    def evaluate(self, freqs):
        return np.ones((len(np.atleast_1d(freqs)), 1))


C_HFSS, sh_err = bm.fit_spherical_harmonics(FIELD, LMAX)
print('HFSS field truncation error at lmax=%d: %.2f%%' % (LMAX, 100 * float(np.ravel(sh_err)[0])))
FIT = bm.JointBeamFit(DATA1, MODEL, PARAMS, LMAX, ConstantSpectrum(), C_HFSS.reshape(1, -1))
coeff_train, gain_train, pred_train, info_train = FIT.fit(TRAIN, 0.0, maxiter=5000)
coeff_all, gain_all, pred_all, info_all = FIT.fit(GOOD, 0.0, initial=coeff_train,
                                                  initial_gain=gain_train, maxiter=5000)
SINGLE_HELDOUT, SINGLE = pred_train[0], pred_all[0]
for label, info in (('train', info_train), ('all-sample', info_all)):
    print('%s fit: %s iterations, %s' % (label, info.get('iterations', info.get('nit')),
                                         info.get('message')))


def frac_rms(model, use):
    return np.linalg.norm((DATA - model)[use]) / np.linalg.norm(DATA[use])


rows = []
for name, model, held in [
        ('HFSS (one gain)', HFSS, HFSS),
        ('v0013 DPSS, 64 teeth', diag['dpss']['empirical_exported'][i], diag['dpss']['empirical_heldout'][i]),
        ('v0013 PCA, 64 teeth', diag['pca']['empirical_exported'][i], diag['pca']['empirical_heldout'][i]),
        ('single-channel lmax=8, no prior', SINGLE, SINGLE_HELDOUT)]:
    rows.append(dict(model=name, train=frac_rms(held, TRAIN),
                     validation_heldout=frac_rms(held, GOOD & (SPLIT == 1)),
                     test_heldout=frac_rms(held, GOOD & (SPLIT == 2)),
                     all_samples=frac_rms(model, GOOD)))
SCORES = pd.DataFrame(rows).set_index('model')
display(SCORES.round(4))

# The same scores as v0013's own metrics.csv for this tooth, as a check on the reconstruction.
metrics = pd.read_csv(PRODUCT / 'metrics.csv')
mine = metrics[(metrics.channel == CHANNEL) & metrics.model.isin(['hfss', 'dpss_heldout', 'dpss_exported',
                                                                   'pca_heldout', 'pca_exported'])]
display(mine.pivot_table(index='model', columns='split', values='fractional_rms').round(4))

In [ ]:
# One arm constrains only E.e. The transverse component orthogonal to e never reaches
# the data, so an unregularized fit lets it drift; compare the two powers.
theta, phi, e_both = MODEL.frame(DATA1, PARAMS)
Y = bm.real_spherical_harmonics(theta, phi, LMAX)
e_arm = e_both[DATA1.arms[0]]
body = np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], -1)


def pol_power(coeff):
    E = (coeff.reshape(3, -1) @ Y.T).T[GOOD]
    b = body[GOOD]
    E = E - np.sum(E * b, axis=1)[:, None] * b
    e_hat = e_arm[GOOD] / np.linalg.norm(e_arm[GOOD], axis=1)[:, None]
    cross = np.cross(b, e_hat)
    return np.mean(abs(np.sum(E * e_hat, axis=1)) ** 2), np.mean(abs(np.sum(E * cross, axis=1)) ** 2)


pol = pd.DataFrame([dict(model=name, co_polar=co, cross_polar=x, cross_over_co=x / co)
                    for name, c in (('HFSS start', C_HFSS.ravel()), ('single-channel fit', coeff_all.ravel()))
                    for co, x in [pol_power(c)]]).set_index('model')
display(pol.round(4))

In [ ]:
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False, 'axes.spines.right': False})
AZ = DATA1.az_deg + MODEL.az_offset_deg + PARAMS[1]           # beam-frame azimuth
az_plot = (AZ + 180.0) % 360.0 - 180.0
EL = DATA1.el_deg + PARAMS[2]
time = pd.to_datetime(DATA1.t, unit='s', utc=True)
lo, hi = np.percentile(DATA[GOOD & (DATA > 0)], [1, 99.5])
norm = LogNorm(vmin=lo, vmax=hi)
rlim = np.percentile(np.abs((DATA - HFSS)[GOOD]), 98)


def plot_maps(model, name):
    fig, axes = plt.subplots(1, 3, figsize=(19, 7), sharex=True, sharey=True, constrained_layout=True)
    # LogNorm drops values <= 0 (tooth - background is slightly negative in some nulls):
    # clip the log panels at the colour floor so data and model show the same samples.
    panels = [('Data (tooth − background)', np.clip(DATA, lo, None), dict(cmap='viridis', norm=norm)),
              (name, np.clip(model, lo, None), dict(cmap='viridis', norm=norm)),
              ('Residual (data − model)', DATA - model, dict(cmap='RdBu_r', vmin=-rlim, vmax=rlim))]
    for ax, (title, value, kw) in zip(axes, panels):
        sc = ax.scatter(az_plot[GOOD], EL[GOOD], c=value[GOOD], s=4, **kw)
        ax.set_title(title)
        ax.set_xlabel('Azimuth, beam frame [deg]')
        fig.colorbar(sc, ax=ax, orientation='horizontal', pad=0.1, shrink=0.85, label='Counts')
    axes[0].set_ylabel('Elevation [deg]')
    fig.suptitle('%s, %.3f MHz, 07-17 raster' % (name, DATA1.freqs_mhz[0]), fontsize=17)
    plt.show()


def plot_time(models):
    fig, ax = plt.subplots(figsize=(16, 6.5), constrained_layout=True)
    ax.plot(time[GOOD], DATA[GOOD], '.', ms=2, color='0.25', label='data')
    for (name, model), colour in zip(models, ('tab:red', 'tab:green')):
        ax.plot(time[GOOD], (DATA - model)[GOOD], '.', ms=2, color=colour, label='residual, ' + name)
    ax.axhline(0, color='0.6', lw=1)
    ax.set_ylabel('Counts')
    ax.set_xlabel('Time [UTC]')
    ax.set_title('%.3f MHz through the raster' % DATA1.freqs_mhz[0])
    ax.legend(loc='upper right', markerscale=6)
    plt.show()

## Maps: HFSS, the v0013 DPSS beam and the single-channel fit

Same samples and colour scales in every panel; the residual scale is set by HFSS.

In [ ]:
plot_maps(HFSS, 'HFSS (one gain)')
plot_maps(diag['dpss']['empirical_exported'][i], 'v0013 DPSS beam')
plot_maps(SINGLE, 'single-channel ℓmax=8, no prior')
plot_time([('v0013 DPSS', diag['dpss']['empirical_exported'][i]), ('single-channel', SINGLE)])

## Findings

- **The reconstruction matches v0013 exactly.** The HFSS baseline rebuilt here from the v0004 release and v0013's geometry equals v0013's to machine precision, and the HFSS, DPSS and PCA scores in the table equal v0013's `metrics.csv` rows for channel 640.
- **At 156.25 MHz the reference beam beats HFSS only on the test stripes** (DPSS and PCA 0.064 held-out, HFSS 0.075); on the validation stripes HFSS is better (0.067 against 0.074 and 0.073). The reference beam's advantage is across the band, not at every tooth.
- **The single-channel fit has the lowest held-out residual** (0.063 validation, 0.049 test), but it does not give a usable beam. One arm constrains only the co-polar coupling, and with no prior the cross-polar power drifts to about 23 times the co-polar power (HFSS: equal). The multi-frequency fit, with both arms and the HFSS prior, is the beam to use.
- **What the single-channel fit says about the model:** freeing the beam shape at this one frequency lowers the held-out rms by 15% (validation) and 23% (test) relative to the reference beam. The residual left after it keeps the same streaks along elevation at fixed azimuth that the HFSS and reference-beam residuals show, so it is not beam shape that an $\ell_{\max}=8$ field at one frequency can absorb. Its cause (pointing, the tooth background, or something varying in time) is untested here.
- **Caveats:** the stripes are v0013's, not a fresh blind test; the geometry corrections are v0013's fitted values, not refitted here; the transmitter offsets sit at their ±1.06 m bound, which was set from the historical v0001 bracket.